In [2]:
import pandas as pd
import pygeohash as pgh
from tqdm import tqdm

In [3]:
DF20_train_path = "../data/FungiCLEF2023_train_metadata_PRODUCTION.csv" # This is the dev training set page
DF21_val_path = "../data/FungiCLEF2023_val_metadata_PRODUCTION.csv" # This is the "validation" set on the page. We want to use this with additional data for unknown classes
public_test_path = "../FungiCLEF2024_TestMetadata (2).csv" # Public test set
IMG_PATH = "../data/DF_FULL"

DF20_df = pd.read_csv(DF20_train_path)
DF21_df = pd.read_csv(DF21_val_path)
test_df = pd.read_csv(public_test_path)

In [4]:
len(test_df)

40216

In [4]:
DF20_df.rename({'observationID': 'observation_id'}, inplace=True, axis=1)
DF21_df.rename({'observationID': 'observation_id'}, inplace=True, axis=1)

In [6]:
cols = ['observation_id', 'month', 'day', 'Latitude', 'Longitude', 'Habitat', 'Substrate', 'MetaSubstrate',  'species', 'image_path', 'class_id', 'poisonous']


In [7]:
train_df = DF20_df[cols]
valtest_df = DF21_df[cols]

In [6]:
_unknowns = DF21_df[DF21_df.class_id==-1].species.value_counts()

In [7]:
unknown_mapping = {v: k + 1603 for k, v in enumerate(_unknowns[_unknowns > 40].index)}

In [8]:
for ix, row in valtest_df.iterrows():
    if row['class_id'] == -1:
        valtest_df.loc[ix, 'class_id'] = unknown_mapping.get(row['species'], -1)

In [9]:
from sklearn.model_selection import train_test_split

In [8]:
_1_species = valtest_df.species.value_counts().index[valtest_df.species.value_counts()==1]

In [12]:
valtest_df[valtest_df.species.isin(_1_species)].poisonous.sum()

0

In [13]:
_train, _test = train_test_split(valtest_df[~valtest_df.species.isin(_1_species)], test_size=0.5, stratify=valtest_df[~valtest_df.species.isin(_1_species)].species)

NameError: name 'train_test_split' is not defined

In [14]:
_train = pd.concat((train_df, _train, valtest_df[valtest_df.species.isin(_1_species)]))

NameError: name '_train' is not defined

In [14]:
emb = pd.read_parquet("/home/chris/fungiclef-2024/scripts/dinov2_1024.pq")

In [15]:
emb['image_path'] = [i.replace("jpg", "JPG") for i in emb.image_path]

In [17]:
_train = _train[_train.image_path.isin(emb.image_path)]

In [ ]:
_test = _test[_test.image_path.isin(emb.image_path)]

In [25]:
import pickle
mp = pickle.load(open("production_metadata_categorical_columns_mapping.pkl", 'rb'))

In [26]:
mp.keys()

dict_keys(['locality', 'level0Gid', 'level1Gid', 'level2Gid', 'Substrate', 'Habitat', 'MetaSubstrate', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species'])

In [ ]:
import pygeohash as pgh
from tqdm import tqdm

In [27]:
df = test_df

In [110]:
import numpy as np
import math
def process_df(df):
        
    base32 = '0123456789bcdefghjkmnpqrstuvwxyz'

    geohash_list = []
    gh_encoded = []
    for _, row in tqdm(df.iterrows()):
        gh = pgh.encode(row.Latitude, row.Longitude)
        geohash_list.append(geohash_base32_to_int(gh[2:5]) / 32**3)
        gh_encoded.append([base32.index(g)/ 32 for g in gh[1:7]])

    geo = pd.DataFrame(np.array(gh_encoded), columns=[f"g{i}" for i in range(6)], index=df.index)
    geo['g_float'] = geohash_list

    df['month'] = df['month'].fillna(1)
    df['day'] = df['day'].fillna(1)
    df['substrate'] = df['Substrate'].apply(lambda x: mp['Substrate'].get(x, 30))
    df['metasubstrate'] = df['MetaSubstrate'].apply(lambda x: mp['MetaSubstrate'].get(x, 9))
    df['habitat'] = df['Habitat'].apply(lambda x: mp['Habitat'].get(x, 31))

    df['m0'] = df['month'].apply(lambda x: math.sin(2* math.pi * (int(x) - 1) / 12) if x else 0)
    df['m1'] = df['month'].apply(lambda x: math.cos(2* math.pi * (int(x) - 1) / 12) if x else 0)
    df['d0'] = df['day'].apply(lambda x: math.sin(2* math.pi * (int(x) - 1) / 31) if x else 0)
    df['d1'] = df['day'].apply(lambda x: math.cos(2* math.pi * (int(x) - 1) / 31) if x else 0)
    
    substrate_onehot = pd.get_dummies(df.substrate, prefix="substrate").astype('int')
    metasubstrate_onehot = pd.get_dummies(df.metasubstrate, prefix="metasubstrate").astype('int')
    habitat_onehot = pd.get_dummies(df.habitat, prefix="habitat").astype('int')

    metadata_df = df[['observation_id', 'image_path', 'm0', 'm1', 'd0', 'd1',]].join(geo).join(substrate_onehot).join(metasubstrate_onehot).join(habitat_onehot)

    return metadata_df

def geohash_base32_to_int(geohash):
    base32 = '0123456789bcdefghjkmnpqrstuvwxyz'
    num = 0
    for char in geohash:
        num = num * 32 + base32.index(char)
    return num



In [104]:
_train = _train.reset_index().drop('index', axis=1)
_val = _test.reset_index().drop('index', axis=1)
test_df = test_df.reset_index().drop('index', axis=1)

In [118]:
metadata_train =  process_df(_train)
metadata_train.join(_train[['class_id', 'poisonous']]).to_csv('../train.csv', index=False)

325495it [00:10, 31659.32it/s]


In [119]:
metadata_val =  process_df(_val)
metadata_val.join(_val[['class_id', 'poisonous']]).to_csv('../val.csv', index=False)

30305it [00:00, 32324.26it/s]


In [122]:
metadata_val[:1000].to_csv('../trial_submission.csv', index=False)

In [121]:
process_df(test_df).to_csv('../test_preprocessed.csv', index=False)

40216it [00:01, 33669.83it/s]


In [17]:
metadata_val.keys()

NameError: name 'metadata_val' is not defined

In [126]:
pd.read_csv('../test_preprocessed.csv')

,observation_id,image_path,m0,m1,d0,d1,g0,g1,g2,g3,...,habitat_22,habitat_23,habitat_24,habitat_25,habitat_26,habitat_27,habitat_28,habitat_29,habitat_30,habitat_31
0,4100095051,0-4100095051.JPG,0.000000,1.000000,0.000000,1.000000,0.03125,0.96875,0.28125,0.00000,...,0,0,0,0,0,0,0,0,0,0
1,4100091091,0-4100091091.JPG,0.000000,1.000000,0.000000,1.000000,0.09375,0.43750,0.25000,0.96875,...,0,0,0,0,0,0,0,0,0,0
2,4100091095,0-4100091095.JPG,0.000000,1.000000,0.000000,1.000000,0.12500,0.62500,0.46875,0.90625,...,0,0,0,0,0,0,0,0,0,0
3,4100095065,0-4100095065.JPG,0.000000,1.000000,0.000000,1.000000,0.12500,0.65625,0.25000,0.59375,...,0,0,0,0,0,0,0,0,0,0
4,4100095065,1-4100095065.JPG,0.000000,1.000000,0.000000,1.000000,0.12500,0.65625,0.25000,0.59375,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40211,4500000760,0a6bc99d-c849-4a8b-8da3-3fbae5dc5d33.jpg,0.500000,-0.866025,0.485302,-0.874347,0.06250,0.43750,0.56250,0.31250,...,0,0,0,0,0,0,0,0,0,1
40212,4500000710,07e4da23-a2c0-4782-bc5f-e13eb20a6dc9.jpg,-0.866025,-0.500000,-0.394356,0.918958,0.00000,0.00000,0.00000,0.00000,...,0,0,0,0,0,0,0,0,0,1
40213,4500000708,a459431f-c956-4012-96b1-dca49ae13a87.jpg,-0.866025,-0.500000,-0.394356,0.918958,0.00000,0.00000,0.00000,0.00000,...,0,0,0,0,0,0,0,0,0,1
40214,4500000708,a459431f-c956-4012-96b1-dca49ae13a87.jpg,-0.866025,-0.500000,-0.394356,0.918958,0.00000,0.00000,0.00000,0.00000,...,0,0,0,0,0,0,0,0,0,1


In [89]:
df = _train

base32 = '0123456789bcdefghjkmnpqrstuvwxyz'

geohash_list = []
gh_encoded = []
for _, row in tqdm(df.iterrows()):
    gh = pgh.encode(row.Latitude, row.Longitude)
    geohash_list.append(geohash_base32_to_int(gh[2:5]) / 32**3)
    gh_encoded.append([base32.index(g)/ 32 for g in gh[1:7]])

geo = pd.DataFrame(np.array(gh_encoded), columns=[f"g{i}" for i in range(6)], index=df.index)
geo['g_float'] = geohash_list

df['month'] = df['month'].fillna(1)
df['day'] = df['day'].fillna(1)
df['substrate'] = df['Substrate'].apply(lambda x: mp['Substrate'].get(x, 30))
df['metasubstrate'] = df['MetaSubstrate'].apply(lambda x: mp['MetaSubstrate'].get(x, 9))
df['habitat'] = df['Habitat'].apply(lambda x: mp['Habitat'].get(x, 31))

df['m0'] = df['month'].apply(lambda x: math.sin(2* math.pi * (int(x) - 1) / 12) if x else 0)
df['m1'] = df['month'].apply(lambda x: math.cos(2* math.pi * (int(x) - 1) / 12) if x else 0)
df['d0'] = df['day'].apply(lambda x: math.sin(2* math.pi * (int(x) - 1) / 31) if x else 0)
df['d1'] = df['day'].apply(lambda x: math.cos(2* math.pi * (int(x) - 1) / 31) if x else 0)

substrate_onehot = pd.get_dummies(df.substrate, prefix="substrate").astype('int')
metasubstrate_onehot = pd.get_dummies(df.metasubstrate, prefix="metasubstrate").astype('int')
habitat_onehot = pd.get_dummies(df.habitat, prefix="habitat").astype('int')



325495it [00:14, 22335.13it/s]


In [94]:
metadata_df = df[['observation_id', 'image_path', 'm0', 'm1', 'd0', 'd1']].join(geo).join(substrate_onehot).join(metasubstrate_onehot).join(habitat_onehot)


In [98]:
geo

,g0,g1,g2,g3,g4,g5,g_float
0,0.12500,0.62500,0.25000,0.18750,0.28125,0.53125,0.632996
1,0.09375,0.31250,0.78125,0.31250,0.28125,0.06250,0.337219
2,0.12500,0.62500,0.75000,0.15625,0.56250,0.03125,0.648590
3,0.09375,0.31250,0.62500,0.71875,0.40625,0.18750,0.332733
4,0.03125,0.84375,0.84375,0.15625,0.40625,0.90625,0.870270
...,...,...,...,...,...,...,...
59589,0.09375,0.43750,0.28125,0.40625,0.12500,0.71875,0.446686
59602,0.09375,0.43750,0.28125,0.40625,0.12500,0.00000,0.446686
59664,0.09375,0.31250,0.40625,0.21875,0.37500,0.37500,0.325409
59692,0.09375,0.31250,0.15625,0.81250,0.71875,0.09375,0.318176


In [97]:
df[['observation_id', 'image_path', 'm0', 'm1', 'd0', 'd1']]

,observation_id,image_path,m0,m1,d0,d1
0,2238546328,2238546328-30620.JPG,1.000000,6.123234e-17,0.101168,-0.994869
1,2558871973,2558871973-53941.JPG,0.000000,1.000000e+00,0.394356,0.918958
2,2238503501,2238503501-245559.JPG,-0.500000,-8.660254e-01,-0.897805,-0.440394
3,2446759075,2446759075-197643.JPG,-1.000000,-1.836970e-16,-0.937752,0.347305
4,2238472345,2238472345-167057.JPG,-0.500000,-8.660254e-01,-0.790776,-0.612106
...,...,...,...,...,...,...
59589,3380887400,0-3380887400.JPG,-0.866025,-5.000000e-01,-0.571268,0.820763
59602,3380887439,0-3380887439.JPG,-1.000000,-1.836970e-16,0.000000,1.000000
59664,3382561357,0-3382561357.JPG,-1.000000,-1.836970e-16,0.571268,0.820763
59692,3382561400,0-3382561400.JPG,-1.000000,-1.836970e-16,0.571268,0.820763


In [93]:
df[['observation_id', 'image_path', 'm0', 'm1', 'd0', 'd1',]]

,observation_id,image_path,m0,m1,d0,d1
0,2238546328,2238546328-30620.JPG,1.000000,6.123234e-17,0.101168,-0.994869
1,2558871973,2558871973-53941.JPG,0.000000,1.000000e+00,0.394356,0.918958
2,2238503501,2238503501-245559.JPG,-0.500000,-8.660254e-01,-0.897805,-0.440394
3,2446759075,2446759075-197643.JPG,-1.000000,-1.836970e-16,-0.937752,0.347305
4,2238472345,2238472345-167057.JPG,-0.500000,-8.660254e-01,-0.790776,-0.612106
...,...,...,...,...,...,...
59589,3380887400,0-3380887400.JPG,-0.866025,-5.000000e-01,-0.571268,0.820763
59602,3380887439,0-3380887439.JPG,-1.000000,-1.836970e-16,0.000000,1.000000
59664,3382561357,0-3382561357.JPG,-1.000000,-1.836970e-16,0.571268,0.820763
59692,3382561400,0-3382561400.JPG,-1.000000,-1.836970e-16,0.571268,0.820763


In [ ]:
.join(substrate_onehot).join(metasubstrate_onehot).join(habitat_onehot)